# 商品表与品类翻译第一步：粒度、主键和映射覆盖

本次先读取原始字符串，检查商品主键与品类翻译。不删除品类缺失的商品，不自动补齐翻译；数值属性清洗在下一步进行。

商品表一行一个商品；订单商品明细表一行一条订单中的商品明细。同一商品可在多条销售明细出现。成交价格和运费仍来自订单商品明细，商品表没有成交价格字段。

| 商品表字段 | 含义 |
| --- | --- |
| product_id | 商品 ID |
| product_category_name | 葡萄牙语品类名 |
| product_name_lenght | 商品名称字符数，不是名称文本 |
| product_description_lenght | 商品描述字符数，不是描述文本 |
| product_photos_qty | 商品展示图片数量 |
| product_weight_g | 记录的重量，单位克 |
| product_length_cm | 记录的长度，单位厘米 |
| product_height_cm | 记录的高度，单位厘米 |
| product_width_cm | 记录的宽度，单位厘米 |

字段解释参照原始目录的中文数据字典。两个 lenght 是原始字段拼写，读取时按原名使用。翻译表翻译的是品类，不是每个商品的名称。

In [19]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path(r'D:\数据分析\olist-operations-analytics')
data_dir = project_dir.parent / 'archive'

def read_raw(filename):
    # ID、品类和原始数值先按字符串读取；后续再检查和转换。
    # 原始表保留，下面的关联结果使用新的变量。
    return pd.read_csv(
        data_dir / filename,
        dtype=str,
        encoding='utf-8-sig',
        keep_default_na=False,
        na_values=['']
    )

products_raw = read_raw('olist_products_dataset.csv')
translation_raw = read_raw('product_category_name_translation.csv')

print('商品表行数、列数：', products_raw.shape)
print('翻译表行数、列数：', translation_raw.shape)
display(products_raw.head(3))
display(translation_raw.head(3))

商品表行数、列数： (32951, 9)
翻译表行数、列数： (71, 2)


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [20]:
# 这里只盘点缺失与类型，不把缺失重量、尺寸或展示信息填为零。
products_raw.info()
products_raw.isna().sum().sort_values(ascending=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   product_id                  32951 non-null  object
 1   product_category_name       32341 non-null  object
 2   product_name_lenght         32341 non-null  object
 3   product_description_lenght  32341 non-null  object
 4   product_photos_qty          32341 non-null  object
 5   product_weight_g            32949 non-null  object
 6   product_length_cm           32949 non-null  object
 7   product_height_cm           32949 non-null  object
 8   product_width_cm            32949 non-null  object
dtypes: object(9)
memory usage: 2.3+ MB


product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
product_id                      0
dtype: int64

## 1. 商品主键与翻译映射键

商品表主键候选是 product_id；翻译表映射键是 product_category_name。

同一品类可以包含多个商品，因此品类名在商品表里可以重复。不能把这种重复当作重复商品直接删除。

In [21]:
product_key_check = pd.Series({
    '商品记录行数': len(products_raw),
    'product_id不同值数': products_raw['product_id'].nunique(),
    'product_id缺失行数': products_raw['product_id'].isna().sum(),
    'product_id非空但空白行数': (
        products_raw['product_id'].notna()
        & products_raw['product_id'].str.strip().eq('')
    ).sum(),
    'product_id重复行数_首条之后': products_raw['product_id'].duplicated().sum(),
    '整行重复数_首条之后': products_raw.duplicated().sum()
})

translation_key_check = pd.Series({
    '翻译记录行数': len(translation_raw),
    '原始品类键不同值数': translation_raw['product_category_name'].nunique(),
    '原始品类键缺失行数': translation_raw['product_category_name'].isna().sum(),
    '原始品类键非空但空白行数': (
        translation_raw['product_category_name'].notna()
        & translation_raw['product_category_name'].str.strip().eq('')
    ).sum(),
    '原始品类键重复行数_首条之后': (
        translation_raw['product_category_name'].duplicated().sum()
    ),
    '英文品类名缺失行数': translation_raw['product_category_name_english'].isna().sum(),
    '英文品类名非空但空白行数': (
        translation_raw['product_category_name_english'].notna()
        & translation_raw['product_category_name_english'].str.strip().eq('')
    ).sum()
})

display(product_key_check)
display(translation_key_check)

商品记录行数                 32951
product_id不同值数         32951
product_id缺失行数             0
product_id非空但空白行数          0
product_id重复行数_首条之后        0
整行重复数_首条之后                 0
dtype: int64

翻译记录行数            71
原始品类键不同值数         71
原始品类键缺失行数          0
原始品类键非空但空白行数       0
原始品类键重复行数_首条之后     0
英文品类名缺失行数          0
英文品类名非空但空白行数       0
dtype: int64

In [22]:
# 连接前验证商品主键，避免同一个 product_id 出现多条属性记录。
if (
    products_raw['product_id'].isna().any()
    or products_raw['product_id'].str.strip().eq('').any()
    or products_raw['product_id'].duplicated().any()
):
    raise ValueError('商品主键缺失、空白或重复，请先核查')

# 右侧映射键必须非空且唯一，才能保证每个商品最多匹配一条翻译。
mapping_key = translation_raw['product_category_name']
english_value = translation_raw['product_category_name_english']

if (
    mapping_key.isna().any()
    or mapping_key.str.strip().eq('').any()
    or mapping_key.duplicated().any()
    or english_value.isna().any()
    or english_value.str.strip().eq('').any()
):
    raise ValueError('翻译表存在缺失、空白或重复映射键，请先核查')

# 同一个英文名称可以用于不同原始品类；多对一关联只要求右侧原始品类键唯一。
print('连接前的主键和映射检查通过')

连接前的主键和映射检查通过


In [23]:
# 品类在商品表里不是主键：它可以重复，也可能缺失。
source_category = products_raw['product_category_name']

pd.Series({
    '商品表有值的不同品类数': source_category.nunique(),
    '原始品类缺失商品数': source_category.isna().sum(),
    '原始品类非空但只有空白商品数': (
        source_category.notna() & source_category.str.strip().eq('')
    ).sum(),
    '翻译表原始品类键数': mapping_key.nunique()
})

商品表有值的不同品类数        73
原始品类缺失商品数         610
原始品类非空但只有空白商品数      0
翻译表原始品类键数          71
dtype: int64

## 2. 为什么用 many_to_one 左连接

左侧同一品类有多个商品；右侧每个原始品类只有一条映射。因此允许左侧多条记录匹配右侧一条，使用 validate='many_to_one'。

how='left' 保留全部商品。validate 只检查映射关系，不负责去重。两个来源都保留原始值，关联结果另存为新表。

In [24]:
products_with_categories = products_raw.merge(
    translation_raw,
    on='product_category_name',
    how='left',
    validate='many_to_one',
    indicator='category_match'
)

# 缺失和纯空白都表示原始品类不可用；原始列本身仍保留。
products_with_categories['flag_category_missing'] = (
    products_with_categories['product_category_name'].isna()
    | products_with_categories['product_category_name'].str.strip().eq('')
)

# 只有原始品类有值、但左连接找不到映射时，才属于“有品类但未翻译”。
products_with_categories['flag_category_untranslated'] = (
    ~products_with_categories['flag_category_missing']
    & products_with_categories['category_match'].eq('left_only')
)

category_coverage_check = pd.Series({
    '关联后商品行数': len(products_with_categories),
    '成功匹配翻译商品数': products_with_categories['category_match'].eq('both').sum(),
    '没有匹配翻译商品数': products_with_categories['category_match'].eq('left_only').sum(),
    '原始品类缺失商品数': products_with_categories['flag_category_missing'].sum(),
    '原始品类有值但未翻译商品数': (
        products_with_categories['flag_category_untranslated'].sum()
    ),
    '英文品类缺失商品数': (
        products_with_categories['product_category_name_english'].isna().sum()
    )
})

display(category_coverage_check)
products_with_categories['category_match'].value_counts()

关联后商品行数          32951
成功匹配翻译商品数        32328
没有匹配翻译商品数          623
原始品类缺失商品数          610
原始品类有值但未翻译商品数       13
英文品类缺失商品数          623
dtype: int64

category_match
both          32328
left_only       623
right_only        0
Name: count, dtype: int64

In [25]:
# 查看原始品类有值但无翻译的类别；暂不自动翻译、归为其他或删除商品。
untranslated_categories = products_with_categories.loc[
    products_with_categories['flag_category_untranslated'],
    'product_category_name'
].value_counts()

display(untranslated_categories)

# 翻译表有但当前商品未使用的映射只做覆盖观察，不当作重复错误。
unused_translation_count = (
    ~translation_raw['product_category_name'].isin(
        products_raw['product_category_name'].dropna()
    )
).sum()
print('当前商品未使用的翻译映射数：', unused_translation_count)

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

当前商品未使用的翻译映射数： 0


## 对照基线

商品表 32,951 行，product_id 非空且唯一。翻译表 71 行，原始品类键非空且唯一；商品表有值的品类共 73 个。

左连接后商品仍 32,951 行。英文品类成功匹配 32,328 个商品，未匹配 623 个：原始品类缺失 610 个，原始品类有值但未翻译 13 个。

13 个商品涉及 portateis_cozinha_e_preparadores_de_alimentos（10 个）和 pc_gamer（3 个）。两类缺失原因保留不同标记，英文名保持缺失。

下一步再检查名称/描述长度、图片数、重量和尺寸，转换数值类型并制定指标有效性条件。

## 我的理解

1. 商品表有多少行、多少个不同 product_id？它的一行与订单商品明细的一行有什么区别？

我的回答：商品表有 32,951 行、32,951 个不同的 product_id，一行代表一个商品的属性信息。订单商品明细的一行代表某笔订单中的一条商品明细，一笔订单可以对应多行。

2. 商品表的主键是什么？翻译表的映射键是什么？product_category_name 能否作为商品表主键？

我的回答：商品表的主键是product_id,翻译表的映射键是product_category_name，不能作为商品表的主键，因为商品表中的product_category_name有重复出现还有缺失。

3. 这次关联为什么使用 many_to_one，而不是 one_to_one？它具体保护哪一侧的唯一性？

我的回答：因为商品表中的同一品类有多个商品，而右侧翻译表中的品类只映射一个英文翻译。

4. 623 个英文品类缺失商品是否都属于“翻译表没有对应项”？应分为哪两类，分别多少个？是否应直接删除这些商品？

我的回答：分类和数量正确。保留这些商品还因为缺少品类或译名，并不代表其他属性及关联销售记录失效。人工核对补充翻译时，应另存新增映射及依据，保留原始翻译表。


In [26]:
numeric_cols = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

# 先生成检查用的数值表，保留此前的品类关联结果。
numeric_values = products_with_categories[numeric_cols].apply(
    pd.to_numeric,
    errors='coerce'
)

numeric_check = pd.DataFrame({
    '原始缺失数': products_with_categories[numeric_cols].isna().sum(),
    '非空但解析失败数': (
        products_with_categories[numeric_cols].notna()
        & numeric_values.isna()
    ).sum(),
    '非有限数': numeric_values.isin(
        [float('inf'), float('-inf')]
    ).sum(),
    '非整数数': (
        numeric_values.notna()
        & numeric_values.mod(1).ne(0)
    ).sum(),
    '负数数': numeric_values.lt(0).sum(),
    '零值数': numeric_values.eq(0).sum()
})

numeric_check

,原始缺失数,非空但解析失败数,非有限数,非整数数,负数数,零值数
product_name_lenght,610,0,0,0,0,0
product_description_lenght,610,0,0,0,0,0
product_photos_qty,610,0,0,0,0,0
product_weight_g,2,0,0,0,0,4
product_length_cm,2,0,0,0,0,0
product_height_cm,2,0,0,0,0,0
product_width_cm,2,0,0,0,0,0


In [27]:
blocking_cols = [
    '非空但解析失败数', '非有限数', '非整数数', '负数数'
]

if numeric_check[blocking_cols].ne(0).any().any():
    raise ValueError('数值检查未通过，请先核查')

products_clean = products_with_categories.copy()

# 当前数据的非空数值均为整数，采用可空整数类型。
# 缺失保持 <NA>，图片数量未知也不会被填成 0。
products_clean[numeric_cols] = numeric_values.astype('Int64')

products_clean['flag_weight_missing'] = (
    products_clean['product_weight_g'].isna()
)

# 重量未知时，此标记保持未知；零重量记录保留并待核查。
products_clean['flag_weight_zero'] = (
    products_clean['product_weight_g'].eq(0)
)

products_clean[numeric_cols].dtypes

product_name_lenght           Int64
product_description_lenght    Int64
product_photos_qty            Int64
product_weight_g              Int64
product_length_cm             Int64
product_height_cm             Int64
product_width_cm              Int64
dtype: object

In [28]:
dimension_cols = [
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

products_clean['can_calculate_volume'] = (
    products_clean[dimension_cols].notna().all(axis=1)
    & products_clean[dimension_cols].gt(0).all(axis=1)
)

# 三边单位都是厘米，乘积单位为立方厘米。
# 不满足条件的商品体积保持未知，而不是填成 0。
products_clean['product_volume_cm3'] = (
    products_clean['product_length_cm']
    * products_clean['product_height_cm']
    * products_clean['product_width_cm']
).where(products_clean['can_calculate_volume'])

print('可计算体积商品数：',
      products_clean['can_calculate_volume'].sum())
print('体积未知商品数：',
      products_clean['product_volume_cm3'].isna().sum())

products_clean['product_volume_cm3'].describe(
    percentiles=[0.5, 0.9, 0.99]
).round(2)

可计算体积商品数： 32949
体积未知商品数： 2


count      32949.0
mean       16564.1
std       27057.04
min          168.0
50%         6840.0
90%        41238.8
99%      135917.44
max       296208.0
Name: product_volume_cm3, dtype: Float64

In [29]:
# fillna(False) 只用于筛选“已知重量记录为0”的商品，
# 不会改写原来的重量或标记。
products_clean.loc[
    products_clean['flag_weight_zero'].fillna(False),
    [
        'product_id',
        'product_weight_g',
        'can_calculate_volume',
        'product_volume_cm3'
    ]
]

,product_id,product_weight_g,can_calculate_volume,product_volume_cm3
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,0,True,22500
13683,8038040ee2a71048d4bdbbdc985b69ab,0,True,22500
14997,36ba42dd187055e1fbe943b2d11430ca,0,True,22500
32079,e673e90efa65a5409ff4196c038bb5af,0,True,22500


In [30]:
products_clean['can_use_weight'] = (
    products_clean['product_weight_g'].notna()
    & products_clean['product_weight_g'].gt(0)
)

print('满足基础重量条件的商品数：',
      products_clean['can_use_weight'].sum())

满足基础重量条件的商品数： 32945


In [31]:
import sys

src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from items_cleaning import clean_order_items

# 复用当前 Notebook 的读取函数和之前完成的明细清洗函数。
# 金额转换为整数分，后面使用 price_cents、freight_cents 核对。
items_raw = read_raw('olist_order_items_dataset.csv')
items_clean = clean_order_items(items_raw)

# 同一商品可以出现在多条销售明细中；
# 右侧商品表必须每个 product_id 只有一行。
items_with_products = items_clean.merge(
    products_clean,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator='products_match'
)

print('关联前明细行数：', len(items_clean))
print('关联后明细行数：', len(items_with_products))

# 关联后仍应保持“一行一条订单商品明细”。
print('关联后组合主键重复数：',
      items_with_products.duplicated(
          ['order_id', 'order_item_id']
      ).sum())

items_with_products['products_match'].value_counts()

关联前明细行数： 112650
关联后明细行数： 112650
关联后组合主键重复数： 0


products_match
both          112650
left_only          0
right_only         0
Name: count, dtype: int64

In [32]:
product_join_check = pd.DataFrame({
    '关联前': [
        len(items_clean),
        items_clean['price_cents'].sum(),
        items_clean['freight_cents'].sum()
    ],
    '关联后': [
        len(items_with_products),
        items_with_products['price_cents'].sum(),
        items_with_products['freight_cents'].sum()
    ]
}, index=['商品明细行数', '商品金额（分）', '运费（分）'])

product_join_check['差额'] = (
    product_join_check['关联后']
    - product_join_check['关联前']
)

if product_join_check['差额'].ne(0).any():
    raise ValueError('关联后数量或金额发生变化，请先核查')

product_join_check

,关联前,关联后,差额
商品明细行数,112650,112650,0
商品金额（分）,1359164370,1359164370,0
运费（分）,225190954,225190954,0
